# EDA for Telco Customer Churn dataset

[dataset link](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
[dataset description](https://community.ibm.com/community/user/blogs/steven-macko/2019/07/11/telco-customer-churn-1113)

In [ ]:
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)

## Download dataset

In [ ]:
%pip install -q kagglehub

In [ ]:
# download dataset from kaggle
import kagglehub # pyright: ignore[reportMissingImports]
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
path = path / r"WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(path)
df.head()

## Prelimitary analys

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
is_na_default = df.isna().sum()
is_na_default.name = "is_na_default"

is_na_space = df.astype(str).map(lambda x: x == "" or x == " ").sum()
is_na_space.name = "is_na_space"

pd.concat([is_na_default, is_na_space], axis = 1)

In [ ]:
df.duplicated().sum()

## Data preparation (basic)

In [ ]:
# to_bool_columns = (
#     ["SeniorCitizen", "Partner", "Dependents", 
#      "PhoneService", "OnlineSecurity", "OnlineBackup", 
#      "DeviceProtection", "TechSupport", "StreamingTV", 
#      "StreamingMovies", "PaperlessBilling", "Churn"])

# def procces_boolean(x):
#     if x == 1 or x == 0:
#         return simple_boolean(x) 
#     else:
#         return text_bool(x)
    
# def simple_boolean(x):
#     return(bool(x))

# def text_bool(x:str):
#     if x.lower() == "no":
#         return False
#     elif x.lower() == "yes":
#         return True
#     else:
#         raise ValueError("Processing an unforeseen variable %s", x)
    
# for columnn in to_bool_columns:
#     df[columnn] = df[columnn].map(lambda x: procces_boolean(x)) 
    

In [ ]:
df.dtypes

In [ ]:
df["TotalCharges"]  = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.describe()

## Data Observation

Check data distribution

In [ ]:
from dataclasses import dataclass
import math


@dataclass
class DiscretePlot():
    data: pd.Series
    title: str
    x_title:str | None = None
    y_title: str| None = None
    
def _count_plot_sizes(len_plot_map: int) -> tuple[int, int]:
    columns = 2 if len_plot_map == 2 else 3
    if len_plot_map < columns:
        columns = len_plot_map
    
    rows = math.ceil(len_plot_map / columns)
    return rows, columns
    
def plot_discrete_vars(plot_map: list[DiscretePlot]):
    nrows, ncolumns = _count_plot_sizes(len(plot_map))
    fig, axs = plt.subplots(nrows, ncolumns, figsize=(5 * ncolumns, 4 * nrows))
    
    if len(plot_map) == 1:
        axs = [axs]
    
    for i, plot in enumerate(plot_map):
        ax = axs.flat[i]
        
        counts = plot.data.value_counts()
        sns.barplot(x=counts.index, y=counts.values, ax=ax)
        
        ax.set_title(plot.title)
        ax.set_xlabel(plot.x_title if plot.x_title else "")
        ax.set_ylabel(plot.y_title if plot.y_title else "Count")
        ax.tick_params(axis='x', labelrotation=45)
    
    total_cells = nrows * ncolumns
    if total_cells > len(plot_map):
        for j in range(len(plot_map), total_cells):
            axs.flat[j].set_visible(False)
    
    print("Proceeded %s maps", len(plot_map))
    plt.tight_layout()
    plt.show()

In [ ]:
dicrete_plot_map = [
    DiscretePlot(df["gender"], title="Customer gender"),
    DiscretePlot(df["SeniorCitizen"], title="Customer senior citizen", x_title="Are customers senior citizens"),
    DiscretePlot(df["Partner"], title="Customer with partner", x_title="Do customers have a partner"),
    DiscretePlot(df["Dependents"], title="Customer having dependents", x_title="Do customers have a dependent"),
    DiscretePlot(df["PhoneService"], title="Customer with phone service", x_title="Do customers include phone service plan"),
    DiscretePlot(df["MultipleLines"], title="Customer with multiple lines"),
    DiscretePlot(df["InternetService"], title="Customer internet service type"),
    DiscretePlot(df["OnlineSecurity"], title="Customer with online security", x_title="Do customers buy online security"),
    DiscretePlot(df["OnlineBackup"], title="Customer with online backup", x_title="Do customers buy online beckup"),
    DiscretePlot(df["DeviceProtection"], title="Customer with device protection", x_title="Do customers buy device protection"),
    DiscretePlot(df["DeviceProtection"], title="Customer with premium tech support", x_title="Do customers buy premium tech support"),
    DiscretePlot(df["StreamingTV"], title="Customer streaming TV", x_title="Do customers stream TV from another providers"),
    DiscretePlot(df["StreamingMovies"], title="Customer streaming movies", x_title="Do customers stream movies from another providers"),
    DiscretePlot(df["Contract"], title="Customer contract types"),
    DiscretePlot(df["PaperlessBilling"], title="Customer use paperless billing"),
    DiscretePlot(df["PaymentMethod"], title="Customer payment method"),
    DiscretePlot(df["Churn"], title="Customer left", x_title="Do customers left this quarter"),
]

plot_discrete_vars(dicrete_plot_map)

In [ ]:
ax = sns.histplot(df.tenure, bins=20, kde=True)
ax.set_title("Tenure density")
plt.show()